In [1]:
# ==========================================
# EduSentinel - Streamlit Application
# ==========================================

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt

st.set_page_config(page_title="EduSentinel", layout="wide")

st.title("🎓 EduGuard")
st.subheader("Explainable AI for Early Academic Risk Detection")

# -----------------------------
# Load Data (for dashboard only)
# -----------------------------
df = pd.read_csv("data/student_data.csv")

df["Risk"] = df["G3"].apply(lambda x: 1 if x < 10 else 0)
df_dashboard = df.copy()

# Drop grade columns
df = df.drop(["G1", "G2", "G3"], axis=1)

# One-hot encoding
df = pd.get_dummies(df, drop_first=True)

# -----------------------------
# Load Model + Features
# -----------------------------
model = joblib.load("model/trained_model.pkl")
feature_columns = joblib.load("model/feature_columns.pkl")

# Sidebar Navigation
menu = st.sidebar.selectbox(
    "Navigation",
    ["Dashboard", "Predict Student Risk"]
)

# ==================================================
# DASHBOARD
# ==================================================
if menu == "Dashboard":

    st.header("📊 Dataset Overview")

    col1, col2 = st.columns(2)

    with col1:
        st.metric("Total Students", len(df_dashboard))
        st.metric("At Risk", df_dashboard["Risk"].sum())
        st.metric("Not At Risk", len(df_dashboard) - df_dashboard["Risk"].sum())

    with col2:
        st.bar_chart(df_dashboard["Risk"].value_counts())

    st.subheader("Feature Correlation")
    st.dataframe(df.corr())

# ==================================================
# PREDICTION
# ==================================================
else:

    st.header("🔍 Predict Student Risk")

    input_data = {}

    for col in feature_columns:
        input_data[col] = st.number_input(col, value=0.0)

    input_df = pd.DataFrame([input_data])

    if st.button("Predict Risk"):

        # Ensure feature alignment
        input_df = input_df.reindex(columns=feature_columns, fill_value=0)

        prob = model.predict_proba(input_df)[0][1]
        prediction = model.predict(input_df)[0]

        if prediction == 1:
            st.error(f"⚠ Student is AT RISK (Probability: {prob:.2f})")
        else:
            st.success(f"✅ Student is NOT AT RISK (Probability: {prob:.2f})")

        # -----------------------------
        # SHAP Explainability
        # -----------------------------
        st.subheader("🔎 Explainable AI (SHAP)")

        if "RandomForest" in str(type(model)) or "XGB" in str(type(model)):
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(input_df)

            fig = plt.figure()
            shap.summary_plot(shap_values[1], input_df, show=False)
            st.pyplot(fig)
        else:
            st.info("SHAP visualization optimized for tree-based models.")

        # -----------------------------
        # Intervention Engine
        # -----------------------------
        st.subheader("💡 Suggested Intervention")

        if "studytime" in input_df.columns and input_df["studytime"].values[0] < 2:
            st.write("• Increase daily study hours.")
        if "absences" in input_df.columns and input_df["absences"].values[0] > 10:
            st.write("• Improve attendance consistency.")
        if "health" in input_df.columns and input_df["health"].values[0] < 3:
            st.write("• Consider health & wellness support.")

2026-02-21 13:05:42.760 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-21 13:05:42.761 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-21 13:05:43.332 
  command:

    streamlit run C:\Users\sagar\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-02-21 13:05:43.333 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-21 13:05:43.333 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-21 13:05:43.334 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-21 13:05:43.335 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn